<h1>Find all the users who were active for 3 consecutive days or more</h1>

In [ ]:
spark = pyspark.sql.SparkSession.builder.getOrCreate()

data = [
    (1, '2024-01-01'),
    (1, '2024-01-02'),
    (1, '2024-01-03'),
    (2, '2024-01-01'),
    (2, '2024-01-03'),
    (3, '2024-01-02'),
    (3, '2024-01-03'),
    (3, '2024-01-04'),
    (4, '2024-01-05'),
    (4, '2024-01-06'),
    (4, '2024-01-07'),
    (4, '2024-01-08'),
]

sf_events = spark.createDataFrame(data, schema=['user_id', 'record_date']) \
                 .withColumn('record_date', col('record_date').cast('date'))

sf_events.show()

In [ ]:
# Import your libraries
import pyspark
from pyspark.sql.functions import col,count,row_number,dense_rank
from pyspark.sql.window import Window

In [ ]:
# Start writing code
window_spec_user_rn=Window.partitionBy("user_id").orderBy(col("record_date").asc())
window_spec_gen_rn=Window.orderBy(col("record_date").asc())
user_rn_df=sf_events.withColumn("user_rn",dense_rank().over(window_spec_user_rn)).withColumn("gen_rn",row_number().over(window_spec_gen_rn)).withColumn("missing_days",(col("gen_rn")-col("user_rn"))).groupBy("user_id",col("gen_rn")-col("user_rn")).agg(count("*").alias("cnt")).filter("cnt==3").select("user_id")


In [ ]:
user_rn_df.show()